# 🤟 ASL Dataset Preprocessing Pipeline
**Dataset:** American Sign Language — kapillondhe

| Bước | Mô tả |
|------|-------|
| 1 | Import & cấu hình |
| 2 | Lọc ảnh: loại mờ, lỗi, trùng lặp |
| 3 | Lưu ảnh: resize 224×224, gán nhãn vào tên file, tạo CSV |
| 4 | Báo cáo, xem mẫu & nén ZIP để tải xuống |

In [1]:
# #======================================================
# #  CELL 1 - Import & Cấu hình                         #
# #======================================================
import os, cv2, hashlib, random, zipfile, warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# -- Đường dẫn ------------------------------------------
TRAIN_DIR  = '/kaggle/input/datasets/kapillondhe/american-sign-language/ASL_Dataset/Train'
OUTPUT_DIR = '/kaggle/working/asl_cleaned'
ZIP_PATH   = '/kaggle/working/asl_cleaned.zip'
CSV_PATH   = os.path.join(OUTPUT_DIR, 'dataset_index.csv')

# -- Tham số --------------------------------------------
SAMPLES_PER_CLASS = 500
BLUR_THRESHOLD    = 5.0   # Phù hợp dataset ASL (max blur ~21)
MIN_SIZE_KB       = 1
TARGET_SIZE       = (224, 224)
SEED              = 42
random.seed(SEED)

# -- Kiểm tra đường dẫn ---------------------------------
assert os.path.exists(TRAIN_DIR), f'FAILED: Không tìm thấy: {TRAIN_DIR}'

classes = sorted([d for d in os.listdir(TRAIN_DIR)
                  if os.path.isdir(os.path.join(TRAIN_DIR, d))])

print(' Import & cấu hình thành công!')
print(f' TRAIN_DIR : {TRAIN_DIR}')
print(f' OUTPUT_DIR: {OUTPUT_DIR}')
print(f' Số lớp   : {len(classes)} -> {classes}')
print(f'  Cấu hình : {SAMPLES_PER_CLASS} ảnh/lớp | blur>={BLUR_THRESHOLD} | resize {TARGET_SIZE}')

 Import & cấu hình thành công!
 TRAIN_DIR : /kaggle/input/datasets/kapillondhe/american-sign-language/ASL_Dataset/Train
 OUTPUT_DIR: /kaggle/working/asl_cleaned
 Số lớp   : 28 -> ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'Nothing', 'O', 'P', 'Q', 'R', 'S', 'Space', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
  Cấu hình : 500 ảnh/lớp | blur>=5.0 | resize (224, 224)


In [2]:
# #======================================================
# #  CELL 2 - Lọc ảnh (sample -> lỗi -> mờ -> dedup)     #
# #======================================================

# -- Bước 2a: Sample 500 ảnh / lớp ----------------------
sampled_dict = {}
for cls in classes:
    all_imgs = [
        os.path.join(TRAIN_DIR, cls, f)
        for f in os.listdir(os.path.join(TRAIN_DIR, cls))
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))
    ]
    sampled_dict[cls] = random.sample(all_imgs, min(SAMPLES_PER_CLASS, len(all_imgs)))

total_sampled = sum(len(v) for v in sampled_dict.values())
print(f' Sau sample: {total_sampled:,} ảnh ({SAMPLES_PER_CLASS}/lớp x {len(classes)} lớp)')

# -- Bước 2b: Lọc & dedup -------------------------------
seen_hashes  = set()
cleaned_dict = {}
report_rows  = []

for cls in tqdm(classes, desc=' Lọc ảnh'):
    paths = sampled_dict[cls]
    good  = []
    n_small = n_corrupt = n_blur = n_dup = 0

    for p in paths:
        # 1) File quá nhỏ
        try:
            if os.path.getsize(p) < MIN_SIZE_KB * 1024:
                n_small += 1; continue
        except OSError:
            n_corrupt += 1; continue

        # 2) Kiểm tra PIL (file lỗi / truncated)
        try:
            with Image.open(p) as img:
                img.verify()
            with Image.open(p) as img:
                img.load()
        except Exception:
            n_corrupt += 1; continue

        # 3) OpenCV đọc được không
        img_cv = cv2.imread(p)
        if img_cv is None:
            n_corrupt += 1; continue

        # 4) Blur score (Laplacian variance)
        gray  = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
        score = cv2.Laplacian(gray, cv2.CV_64F).var()
        if score < BLUR_THRESHOLD:
            n_blur += 1; continue

        # 5) Duplicate (MD5)
        with open(p, 'rb') as f:
            h = hashlib.md5(f.read()).hexdigest()
        if h in seen_hashes:
            n_dup += 1; continue
        seen_hashes.add(h)
        good.append(p)

    cleaned_dict[cls] = good
    report_rows.append({
        'class'  : cls,
        'sample' : len(paths),
        'small'  : n_small,
        'corrupt': n_corrupt,
        'blur'   : n_blur,
        'dup'    : n_dup,
        'kept'   : len(good)
    })

df_report = pd.DataFrame(report_rows)
print('\n Báo cáo lọc:')
print(df_report.to_string(index=False))
print(f"\n Tổng ảnh sạch: {df_report['kept'].sum():,} / {total_sampled:,}")
print(f"  Đã loại     : {total_sampled - df_report['kept'].sum():,} ảnh")

 Sau sample: 14,000 ảnh (500/lớp x 28 lớp)


 Lọc ảnh:   0%|          | 0/28 [00:00<?, ?it/s]


 Báo cáo lọc:
  class  sample  small  corrupt  blur  dup  kept
      A     500      0        0     0    0   500
      B     500      0        0     7    0   493
      C     500      0        0     0    0   500
      D     500      0        0     0    0   500
      E     500      0        0     2    0   498
      F     500      0        0     0    0   500
      G     500      0        0     1    0   499
      H     500      0        0     7    0   493
      I     500      0        0     2    0   498
      J     500      0        0     0    0   500
      K     500      0        0     0    0   500
      L     500      0        0     0    0   500
      M     500      0        0     1    0   499
      N     500      0        0     0    0   500
Nothing     500      0        0   488    0    12
      O     500      0        0     0    0   500
      P     500      0        0     0    0   500
      Q     500      0        0     4    0   496
      R     500      0        0     1    0   499
     

In [3]:
# #======================================================
# #  CELL 3 - Lưu ảnh, gán nhãn vào tên file, tạo CSV  #
# #======================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mapping nhãn -> số nguyên (A=0, B=1, ...)
label_list = sorted(cleaned_dict.keys())
label2id   = {l: i for i, l in enumerate(label_list)}

print('  Mapping nhãn:')
for l, i in label2id.items():
    print(f'   {l:10s} -> {i}')
print()

csv_rows = []

for cls in tqdm(label_list, desc=' Lưu ảnh'):
    cls_out = os.path.join(OUTPUT_DIR, cls)
    os.makedirs(cls_out, exist_ok=True)
    paths = cleaned_dict[cls]

    for idx, p in enumerate(paths):
        try:
            with Image.open(p) as img:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                img = img.resize(TARGET_SIZE, Image.LANCZOS)

                # Tên file = NHÃN_số thứ tự -> nhãn ghi thẳng vào tên
                new_name = f'{cls}_{idx+1:04d}.jpg'
                out_path = os.path.join(cls_out, new_name)
                img.save(out_path, 'JPEG', quality=95)

                csv_rows.append({
                    'filepath' : out_path,
                    'filename' : new_name,
                    'label'    : cls,
                    'label_id' : label2id[cls]
                })
        except Exception as e:
            print(f'  Lỗi lưu {p}: {e}')

# Tạo & lưu CSV
df_final = pd.DataFrame(csv_rows)
df_final.to_csv(CSV_PATH, index=False)

print(f'\n Đã lưu {len(df_final):,} ảnh -> {OUTPUT_DIR}')
print(f' CSV   -> {CSV_PATH}')
print(f'\n Mẫu CSV (5 dòng đầu):')
print(df_final.head().to_string(index=False))

  Mapping nhãn:
   A          -> 0
   B          -> 1
   C          -> 2
   D          -> 3
   E          -> 4
   F          -> 5
   G          -> 6
   H          -> 7
   I          -> 8
   J          -> 9
   K          -> 10
   L          -> 11
   M          -> 12
   N          -> 13
   Nothing    -> 14
   O          -> 15
   P          -> 16
   Q          -> 17
   R          -> 18
   S          -> 19
   Space      -> 20
   T          -> 21
   U          -> 22
   V          -> 23
   W          -> 24
   X          -> 25
   Y          -> 26
   Z          -> 27



 Lưu ảnh:   0%|          | 0/28 [00:00<?, ?it/s]


 Đã lưu 13,472 ảnh -> /kaggle/working/asl_cleaned
 CSV   -> /kaggle/working/asl_cleaned/dataset_index.csv

 Mẫu CSV (5 dòng đầu):
                                filepath   filename label  label_id
/kaggle/working/asl_cleaned/A/A_0001.jpg A_0001.jpg     A         0
/kaggle/working/asl_cleaned/A/A_0002.jpg A_0002.jpg     A         0
/kaggle/working/asl_cleaned/A/A_0003.jpg A_0003.jpg     A         0
/kaggle/working/asl_cleaned/A/A_0004.jpg A_0004.jpg     A         0
/kaggle/working/asl_cleaned/A/A_0005.jpg A_0005.jpg     A         0


---
## 🔀 PHẦN 2: Merge dataset mới (ysjprojects/asl-signs)

**Đặc điểm dataset mới:**
- Chỉ có tay, nền đơn sắc (xám/nâu), ảnh 128×128
- Lớp `BLANK` → map thành `Nothing`
- Không có `Space` → giữ nguyên từ dataset cũ
- **Không cần crop** vì không có mặt người

**Chiến lược:** Mỗi lớp tăng từ 500 → 800 ảnh (thêm tối đa 300 ảnh mới)

In [7]:
# #======================================================
# #  CELL 4 - Load & lọc dataset mới (ysjprojects)      #
# #======================================================
# Dataset mới: ysjprojects/asl-signs
#   - Cấu trúc: dataset/ -> A/, B/, ..., Z/, BLANK/
#   - Ảnh 128x128, chỉ có tay, nền đơn sắc -> KHÔNG cần crop
#   - BLANK = tương đương Nothing -> đổi tên khi merge
#   - KHÔNG có Space -> giữ nguyên Space từ dataset cũ

NEW_TRAIN_DIR = '/kaggle/input/datasets/ysjprojects/asl-signs/dataset'  # <- sửa nếu cần

# Mapping tên lớp dataset mới -> tên chuẩn của pipeline
# BLANK -> Nothing | các chữ A-Z giữ nguyên | Space không có -> bỏ qua
CLASS_REMAP = {'BLANK': 'Nothing'}
SKIP_CLASSES = []  # Lớp nào không muốn lấy từ dataset mới

# Kiểm tra đường dẫn & liệt kê lớp
import os
assert os.path.exists(NEW_TRAIN_DIR), f'FAILED: Không tìm thấy: {NEW_TRAIN_DIR}'

raw_classes_new = sorted([
    d for d in os.listdir(NEW_TRAIN_DIR)
    if os.path.isdir(os.path.join(NEW_TRAIN_DIR, d))
])

print(f' Dataset mới: {NEW_TRAIN_DIR}')
print(f' Lớp thô    : {raw_classes_new}')

# Áp dụng remap
classes_new = []
for c in raw_classes_new:
    mapped = CLASS_REMAP.get(c, c)
    if mapped not in SKIP_CLASSES:
        classes_new.append((c, mapped))  # (tên gốc, tên chuẩn)

print(f'\n Sau remap:')
for orig, mapped in classes_new:
    arrow = f' -> {mapped}' if orig != mapped else ''
    print(f'   {orig}{arrow}')

# Đếm ảnh từng lớp
print(f'\n Số ảnh mỗi lớp (dataset mới):')
for orig, mapped in classes_new:
    n = len(os.listdir(os.path.join(NEW_TRAIN_DIR, orig)))
    print(f'   {mapped:10s}: {n:,} ảnh')


 Dataset mới: /kaggle/input/datasets/ysjprojects/asl-signs/dataset
 Lớp thô    : ['A', 'B', 'BLANK', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']

 Sau remap:
   A
   B
   BLANK -> Nothing
   C
   D
   E
   F
   G
   H
   I
   J
   K
   L
   M
   N
   O
   P
   Q
   R
   S
   T
   U
   V
   W
   X
   Y
   Z

 Số ảnh mỗi lớp (dataset mới):
   A         : 150 ảnh
   B         : 150 ảnh
   Nothing   : 150 ảnh
   C         : 150 ảnh
   D         : 150 ảnh
   E         : 150 ảnh
   F         : 150 ảnh
   G         : 150 ảnh
   H         : 150 ảnh
   I         : 150 ảnh
   J         : 150 ảnh
   K         : 150 ảnh
   L         : 150 ảnh
   M         : 150 ảnh
   N         : 150 ảnh
   O         : 150 ảnh
   P         : 150 ảnh
   Q         : 150 ảnh
   R         : 150 ảnh
   S         : 150 ảnh
   T         : 150 ảnh
   U         : 150 ảnh
   V         : 150 ảnh
   W         : 150 ảnh
   X         : 150 ảnh
   Y     

In [8]:
# #==============================================================
# #  CELL 5 - Lọc dataset mới & merge vào asl_cleaned          #
# #==============================================================
# Chiến lược merge:
#   - Dataset cũ đã lưu vào OUTPUT_DIR (Cell 3) -> giữ nguyên
#   - Dataset mới: lọc blur/corrupt/dedup -> lưu tiếp vào cùng OUTPUT_DIR
#   - Đánh số file tiếp theo sau file cuối cùng của mỗi lớp
#   - Tổng mỗi lớp cố gắng đạt TOTAL_PER_CLASS (mặc định 800)
#   - Lớp Space không có trong dataset mới -> giữ nguyên 500 ảnh cũ

TOTAL_PER_CLASS = 800   # Tổng ảnh mỗi lớp sau merge
                         # (500 cũ + tối đa 300 mới)

import os, cv2, hashlib, random, warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

# -- Đọc CSV cũ để biết file nào đã có -----------------------------
df_old = pd.read_csv(CSV_PATH)
print(f' Dataset cũ: {len(df_old):,} ảnh, {df_old["label"].nunique()} lớp')

# Đếm số ảnh hiện tại mỗi lớp
existing_count = df_old.groupby('label').size().to_dict()
print(f'\n Số ảnh hiện tại mỗi lớp:')
for cls, cnt in sorted(existing_count.items()):
    need = max(0, TOTAL_PER_CLASS - cnt)
    print(f'   {cls:10s}: {cnt:3d} ảnh  -> cần thêm {need:3d}')

# Lấy hash của ảnh cũ để dedup chéo dataset
print(f'\n Đang hash {len(df_old):,} ảnh cũ để tránh trùng lặp...')
seen_hashes = set()
for fp in tqdm(df_old['filepath'], desc='Hash cũ', leave=False):
    try:
        with open(fp, 'rb') as f:
            seen_hashes.add(hashlib.md5(f.read()).hexdigest())
    except:
        pass
print(f' Đã hash {len(seen_hashes):,} ảnh cũ')

# -- Lọc & lưu ảnh từ dataset mới ----------------------------------
new_csv_rows = []
summary_rows = []

for orig_cls, mapped_cls in tqdm(classes_new, desc=' Merge lớp'):
    src_dir = os.path.join(NEW_TRAIN_DIR, orig_cls)
    dst_dir = os.path.join(OUTPUT_DIR, mapped_cls)
    os.makedirs(dst_dir, exist_ok=True)

    # Cần thêm bao nhiêu ảnh?
    current = existing_count.get(mapped_cls, 0)
    need    = max(0, TOTAL_PER_CLASS - current)

    if need == 0:
        print(f'     {mapped_cls}: đã đủ {current} ảnh, bỏ qua')
        summary_rows.append({'class': mapped_cls, 'already': current,
                              'needed': 0, 'added': 0, 'skipped': 0})
        continue

    # Tìm số thứ tự tiếp theo trong thư mục đích
    existing_files = [f for f in os.listdir(dst_dir) if f.endswith('.jpg')]
    next_idx = len(existing_files) + 1

    # Lấy danh sách ảnh nguồn
    all_src_imgs = [
        os.path.join(src_dir, f)
        for f in os.listdir(src_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))
    ]
    random.shuffle(all_src_imgs)

    n_added = n_skip_corrupt = n_skip_blur = n_skip_dup = 0

    for p in all_src_imgs:
        if n_added >= need:
            break

        # 1) Corrupt check
        try:
            with Image.open(p) as img:
                img.verify()
            with Image.open(p) as img:
                img.load()
        except:
            n_skip_corrupt += 1; continue

        img_cv = cv2.imread(p)
        if img_cv is None:
            n_skip_corrupt += 1; continue

        # 2) Blur check (threshold thấp hơn vì ảnh 128px)
        gray  = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
        score = cv2.Laplacian(gray, cv2.CV_64F).var()
        if score < BLUR_THRESHOLD:
            n_skip_blur += 1; continue

        # 3) Dedup chéo 2 dataset
        with open(p, 'rb') as f:
            h = hashlib.md5(f.read()).hexdigest()
        if h in seen_hashes:
            n_skip_dup += 1; continue
        seen_hashes.add(h)

        # 4) Resize & lưu
        try:
            with Image.open(p) as img:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                img = img.resize(TARGET_SIZE, Image.LANCZOS)
                new_name = f'{mapped_cls}_{next_idx:04d}.jpg'
                out_path = os.path.join(dst_dir, new_name)
                img.save(out_path, 'JPEG', quality=95)

                new_csv_rows.append({
                    'filepath' : out_path,
                    'filename' : new_name,
                    'label'    : mapped_cls,
                    'label_id' : label2id[mapped_cls],
                    'source'   : 'ysjprojects'
                })
                next_idx += 1
                n_added  += 1
        except Exception as e:
            n_skip_corrupt += 1; continue

    summary_rows.append({
        'class'  : mapped_cls,
        'already': current,
        'needed' : need,
        'added'  : n_added,
        'skipped': n_skip_corrupt + n_skip_blur + n_skip_dup
    })

# -- Tạo CSV merge -------------------------------------------------
# Thêm cột source vào CSV cũ
df_old['source'] = 'kapillondhe'
df_new_rows = pd.DataFrame(new_csv_rows)
df_merged   = pd.concat([df_old, df_new_rows], ignore_index=True)

MERGED_CSV = os.path.join(OUTPUT_DIR, 'dataset_index_merged.csv')
df_merged.to_csv(MERGED_CSV, index=False)

# -- Báo cáo merge -------------------------------------------------
df_summary = pd.DataFrame(summary_rows)
df_summary['total'] = df_summary['already'] + df_summary['added']

print(f'\n Báo cáo merge:')
print(df_summary.to_string(index=False))
print(f'\n Tổng ảnh sau merge : {len(df_merged):,}')
print(f'   Ảnh từ kapillondhe : {len(df_old):,}')
print(f'   Ảnh từ ysjprojects : {len(df_new_rows):,}')
print(f'   CSV đã lưu         : {MERGED_CSV}')
print(f'\n  Lớp Space: giữ nguyên {existing_count.get("Space",0)} ảnh ')
print(f'   (dataset mới không có Space)')


 Dataset cũ: 13,472 ảnh, 28 lớp

 Số ảnh hiện tại mỗi lớp:
   A         : 500 ảnh  -> cần thêm 300
   B         : 493 ảnh  -> cần thêm 307
   C         : 500 ảnh  -> cần thêm 300
   D         : 500 ảnh  -> cần thêm 300
   E         : 498 ảnh  -> cần thêm 302
   F         : 500 ảnh  -> cần thêm 300
   G         : 499 ảnh  -> cần thêm 301
   H         : 493 ảnh  -> cần thêm 307
   I         : 498 ảnh  -> cần thêm 302
   J         : 500 ảnh  -> cần thêm 300
   K         : 500 ảnh  -> cần thêm 300
   L         : 500 ảnh  -> cần thêm 300
   M         : 499 ảnh  -> cần thêm 301
   N         : 500 ảnh  -> cần thêm 300
   Nothing   :  12 ảnh  -> cần thêm 788
   O         : 500 ảnh  -> cần thêm 300
   P         : 500 ảnh  -> cần thêm 300
   Q         : 496 ảnh  -> cần thêm 304
   R         : 499 ảnh  -> cần thêm 301
   S         : 499 ảnh  -> cần thêm 301
   Space     : 500 ảnh  -> cần thêm 300
   T         : 497 ảnh  -> cần thêm 303
   U         : 498 ảnh  -> cần thêm 302
   V         : 499 ản

Hash cũ:   0%|          | 0/13472 [00:00<?, ?it/s]

 Đã hash 13,472 ảnh cũ


 Merge lớp:   0%|          | 0/27 [00:00<?, ?it/s]


 Báo cáo merge:
  class  already  needed  added  skipped  total
      A      500     300    150        0    650
      B      493     307    150        0    643
Nothing       12     788    129       21    141
      C      500     300    150        0    650
      D      500     300    150        0    650
      E      498     302    150        0    648
      F      500     300    150        0    650
      G      499     301    150        0    649
      H      493     307    150        0    643
      I      498     302    150        0    648
      J      500     300    150        0    650
      K      500     300    150        0    650
      L      500     300    150        0    650
      M      499     301    150        0    649
      N      500     300    150        0    650
      O      500     300    150        0    650
      P      500     300    150        0    650
      Q      496     304    150        0    646
      R      499     301    150        0    649
      S      499     30

---
## PHAN 3: Chia train / val / test (70 / 15 / 15)

Ket qua:
```
asl_split/
    train/  A/ B/ ... Z/ Nothing/ Space/
    val/    A/ B/ ... Z/ Nothing/ Space/
    test/   A/ B/ ... Z/ Nothing/ Space/
    train.csv | val.csv | test.csv
```

In [9]:
# == CELL 7 - Chia train/val/test (70/15/15) ==
import os, shutil, random
import pandas as pd
from tqdm.notebook import tqdm

# -- Cau hinh --
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15   # phan con lai
SEED        = 42
random.seed(SEED)

SPLIT_BASE  = '/kaggle/working/asl_split'
TRAIN_DIR_OUT = os.path.join(SPLIT_BASE, 'train')
VAL_DIR_OUT   = os.path.join(SPLIT_BASE, 'val')
TEST_DIR_OUT  = os.path.join(SPLIT_BASE, 'test')

for d in [TRAIN_DIR_OUT, VAL_DIR_OUT, TEST_DIR_OUT]:
    os.makedirs(d, exist_ok=True)

# -- Doc CSV merged (neu co) hoac CSV cu --
merged_csv = os.path.join(OUTPUT_DIR, 'dataset_index_merged.csv')
base_csv   = os.path.join(OUTPUT_DIR, 'dataset_index.csv')
csv_to_use = merged_csv if os.path.exists(merged_csv) else base_csv
print(f'Doc tu: {csv_to_use}')

df = pd.read_csv(csv_to_use)
print(f'Tong anh: {len(df):,}')
print(f'So lop  : {df["label"].nunique()}')
print()

split_rows = []
label_report = []

for cls in tqdm(sorted(df['label'].unique()), desc='Chia split'):
    subset = df[df['label'] == cls].sample(frac=1, random_state=SEED).reset_index(drop=True)
    n      = len(subset)

    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    n_test  = n - n_train - n_val   # phan con lai tranh mat anh

    splits = {
        'train': subset.iloc[:n_train],
        'val'  : subset.iloc[n_train : n_train + n_val],
        'test' : subset.iloc[n_train + n_val:]
    }
    dirs = {'train': TRAIN_DIR_OUT, 'val': VAL_DIR_OUT, 'test': TEST_DIR_OUT}

    for split_name, split_df in splits.items():
        out_cls_dir = os.path.join(dirs[split_name], cls)
        os.makedirs(out_cls_dir, exist_ok=True)

        for _, row in split_df.iterrows():
            src = row['filepath']
            dst = os.path.join(out_cls_dir, row['filename'])
            if os.path.exists(src):
                shutil.copy2(src, dst)
            split_rows.append({
                'filepath' : dst,
                'filename' : row['filename'],
                'label'    : cls,
                'label_id' : row['label_id'],
                'split'    : split_name
            })

    label_report.append({
        'class': cls, 'total': n,
        'train': len(splits['train']),
        'val'  : len(splits['val']),
        'test' : len(splits['test'])
    })

# -- Luu CSV tung split --
df_split = pd.DataFrame(split_rows)

for split_name in ['train', 'val', 'test']:
    sub = df_split[df_split['split'] == split_name]
    csv_out = os.path.join(SPLIT_BASE, f'{split_name}.csv')
    sub.to_csv(csv_out, index=False)

# -- Bao cao --
df_report = pd.DataFrame(label_report)
print('Bao cao chia split:')
print(df_report.to_string(index=False))
print()

totals = df_report[['train','val','test']].sum()
grand  = totals.sum()
print(f'Tong train : {totals["train"]:,}  ({totals["train"]/grand*100:.1f}%)')
print(f'Tong val   : {totals["val"]:,}   ({totals["val"]/grand*100:.1f}%)')
print(f'Tong test  : {totals["test"]:,}   ({totals["test"]/grand*100:.1f}%)')
print(f'Tong cong  : {grand:,}')
print()
print(f'Thu muc output:')
print(f'   {TRAIN_DIR_OUT}')
print(f'   {VAL_DIR_OUT}')
print(f'   {TEST_DIR_OUT}')
print(f'CSV: {SPLIT_BASE}/train.csv | val.csv | test.csv')


Doc tu: /kaggle/working/asl_cleaned/dataset_index_merged.csv
Tong anh: 17,500
So lop  : 28



Chia split:   0%|          | 0/28 [00:00<?, ?it/s]

Bao cao chia split:
  class  total  train  val  test
      A    650    454   97    99
      B    643    450   96    97
      C    650    454   97    99
      D    650    454   97    99
      E    648    453   97    98
      F    650    454   97    99
      G    649    454   97    98
      H    643    450   96    97
      I    648    453   97    98
      J    650    454   97    99
      K    650    454   97    99
      L    650    454   97    99
      M    649    454   97    98
      N    650    454   97    99
Nothing    141     98   21    22
      O    650    454   97    99
      P    650    454   97    99
      Q    646    452   96    98
      R    649    454   97    98
      S    649    454   97    98
  Space    500    350   75    75
      T    647    452   97    98
      U    648    453   97    98
      V    649    454   97    98
      W    648    453   97    98
      X    645    451   96    98
      Y    648    453   97    98
      Z    650    454   97    99

Tong train : 12,232  (

---
## PHAN 4: Nen output thanh ZIP

In [10]:
# == CELL 8 - Nen toan bo split thanh ZIP ==
import os, zipfile
from tqdm.notebook import tqdm

ZIP_PATH = '/kaggle/working/asl_split.zip'

print('Dang nen thu muc split...')
print(f'   Nguon : {SPLIT_BASE}')
print(f'   Dich  : {ZIP_PATH}')

all_files = []
for root, dirs, files in os.walk(SPLIT_BASE):
    for f in files:
        all_files.append(os.path.join(root, f))

print(f'   Tong  : {len(all_files):,} file')

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in tqdm(all_files, desc='Nen file'):
        arcname = os.path.relpath(fpath, start=os.path.dirname(SPLIT_BASE))
        zf.write(fpath, arcname)

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f'Nen xong!')
print(f'   File ZIP   : {ZIP_PATH}')
print(f'   Kich thuoc : {zip_size_mb:.1f} MB')
print(f'   So file    : {len(all_files):,}')


Dang nen thu muc split...
   Nguon : /kaggle/working/asl_split
   Dich  : /kaggle/working/asl_split.zip
   Tong  : 17,503 file


Nen file:   0%|          | 0/17503 [00:00<?, ?it/s]

Nen xong!
   File ZIP   : /kaggle/working/asl_split.zip
   Kich thuoc : 195.5 MB
   So file    : 17,503
